# Análisis Exploratorio de Datos (EDA) — Amazon Products

Dataset: ~1.5K productos de Amazon con calificaciones y reseñas.

**Objetivo:**
1. Recorrer iterativamente la carpeta `datos/` y consolidar los archivos `.xlsx` en un único DataFrame.
2. Normalizar columnas con nombres heterogéneos (español/inglés) y descartar columnas no documentadas.
3. Auditar calidad, limpiar tipos y realizar análisis univariado y bivariado.
4. Identificar productos y categorías destacadas.

**Columnas oficiales del dataset** (cualquier otra se descarta):
`product_id`, `product_name`, `category`, `discounted_price`, `actual_price`, `discount_percentage`, `rating`, `rating_count`, `about_product`, `user_id`, `user_name`, `review_id`, `review_title`, `review_content`, `img_link`, `product_link`.

In [ ]:
import pandas as pd
from pathlib import Path

from src.carga import cargar_archivos
from src.clustering import (
    clusterizar_kmeans,
    construir_documentos,
    contar_unicas,
    tabla_similitudes,
    vectorizar,
)

In [ ]:
DATA_DIR = Path('datos')
archivos = sorted(DATA_DIR.glob('*.xlsx'))
archivos

In [ ]:
df = cargar_archivos(DATA_DIR)
print(f'Total: {len(df)} filas, {len(df.columns)} columnas')

In [ ]:
df.head()

In [ ]:
df.info()

## Detección automática de columnas equivalentes con K-means

Después del `concat`, las variantes en español/inglés (`categoria` ↔ `category`, `nombre_producto` ↔ `product_name`, `usuario` ↔ `user_name`, `rankeo` ↔ `rating`) viven como columnas separadas en `df`, cada una con valores donde el archivo original las traía y `NaN` en el resto. En lugar de mapearlas a mano, las agrupamos **por su contenido**:

1. Tratamos cada columna de `df` como un *documento*: concatenamos sus valores no nulos en una cadena.
2. Vectorizamos con TF-IDF (peso de cada token según su frecuencia relativa).
3. Aplicamos K-means con `K = 17` (un cluster por concepto esperado).
4. Las columnas que terminan en el mismo cluster son candidatas a representar el mismo campo.

In [ ]:
from collections import defaultdict

In [ ]:
documentos, etiquetas = construir_documentos(df)
print(f'Total de columnas a agrupar: {len(documentos)}')

In [ ]:
X = vectorizar(documentos)
print(f'Matriz TF-IDF: {X.shape[0]} columnas × {X.shape[1]} tokens')

In [ ]:
K = 17
diccionario_clusters = clusterizar_kmeans(X, etiquetas, k=K)

for cluster_id in sorted(diccionario_clusters):
    miembros = diccionario_clusters[cluster_id]
    print(f'Cluster {cluster_id} ({len(miembros)} columna{"s" if len(miembros)!=1 else ""}): {miembros}')

## Conteo dinámico de variables únicas

En lugar de buscar el `K` *óptimo* (silueta, etc.), contamos cuántas variables **realmente distintas** hay en el dataset:

1. Calculamos la similitud coseno por pares entre los vectores TF-IDF de las columnas.
2. Si dos columnas superan un umbral (p. ej. `0.85`) las tratamos como variantes de la **misma** variable.
3. Aplicamos *union-find* sobre los pares fusionados; el número de componentes conectados es el `K_real`.

Excluimos `archivo_origen` por ser metadata que añadimos en la carga, no una columna del dataset original.

**Comportamiento esperado:**
- Con los 4 archivos actuales: `K_real = 17` (16 oficiales + `envio_dias`, fusionando 4 pares ES/EN).
- Si llega un archivo con una columna nueva no equivalente a ninguna existente: `K_real = 18`.
- Si llega una nueva variante de una columna existente (sim ≥ 0.85): `K_real` se mantiene.

In [ ]:
META = {'archivo_origen'}
indices_datos = [i for i, c in enumerate(etiquetas) if c not in META]
X_datos = X[indices_datos]
etiquetas_datos = [etiquetas[i] for i in indices_datos]

UMBRAL_SIMILITUD = 0.85
K_real, pares_fusionados, S = contar_unicas(X_datos, etiquetas_datos, umbral=UMBRAL_SIMILITUD)

print(f'K_real (variables únicas detectadas): {K_real}')
print()
print(f'Pares fusionados (similitud >= {UMBRAL_SIMILITUD}):')
for a, b, s in sorted(pares_fusionados, key=lambda p: -p[2]):
    print(f'  {a:<20} <-> {b:<20}  sim={s:.3f}')

In [ ]:
clusters_reales = clusterizar_kmeans(X_datos, etiquetas_datos, k=K_real)

for cluster_id in sorted(clusters_reales):
    miembros = clusters_reales[cluster_id]
    print(f'Cluster {cluster_id} ({len(miembros)} columna{"s" if len(miembros)!=1 else ""}): {miembros}')

**Cómo se reajusta automáticamente:**

- Añades un archivo `Amazon5.xlsx` con una columna nueva `peso_kg` (sin equivalente): al re-ejecutar las celdas, los TF-IDF se recalculan, no aparece ningún par con `sim ≥ 0.85` para `peso_kg`, y `K_real` pasa a `18`.
- Añades un archivo donde `category` está escrito como `categoría` (mismo contenido): la similitud con la `category` existente quedará `≈ 1.0`, se fusionan, y `K_real` se mantiene en `17`.
- Si quieres que `discounted_price` y `actual_price` se consideren la misma variable (su sim ≈ 0.82), baja el umbral a `0.80`.
- Para incluir `archivo_origen` u otra metadata en el conteo, basta con quitarla del set `META`.

## Estabilidad del umbral: tabla de similitudes por pares

Para juzgar si `0.85` es seguro vemos **todas** las combinaciones de columnas con su similitud y, sobre todo, la **brecha** entre el menor par fusionado y el mayor par no fusionado: cuanto mayor la brecha, menos sensible es el resultado a mover el umbral.

In [ ]:
tabla_similitudes_df = tabla_similitudes(S, etiquetas_datos, umbral=UMBRAL_SIMILITUD)

arriba = tabla_similitudes_df[tabla_similitudes_df['fusiona']]
abajo  = tabla_similitudes_df[~tabla_similitudes_df['fusiona']]
brecha = arriba['similitud'].min() - abajo['similitud'].max()

print(f'Pares totales: {len(tabla_similitudes_df)}')
print(f'Por encima del umbral {UMBRAL_SIMILITUD}: {len(arriba)} pares (min={arriba["similitud"].min():.4f}, max={arriba["similitud"].max():.4f})')
print(f'Por debajo del umbral:               {len(abajo)} pares (max mas cercano={abajo["similitud"].max():.4f})')
print()
print(f'BRECHA = min(fusionados) - max(no fusionados) = {brecha:+.4f}')
print('  > 0.05 -> umbral estable, mover +-0.02 no cambia el resultado')
print('  < 0.02 -> umbral fragil, pequenos cambios reasignan pares')

tabla_similitudes_df

In [ ]:
vecindad = tabla_similitudes_df[
    (tabla_similitudes_df['similitud'] >= 0.70) &
    (tabla_similitudes_df['similitud'] <= 0.95)
].copy()
vecindad['distancia_al_umbral'] = (vecindad['similitud'] - UMBRAL_SIMILITUD).round(4)
vecindad